# IBKR Flex sync

Pulls the "Trade History API" Flex Query (Cash Report + Open Positions + Trades) and brings `data/brokers/ibkr/ledger.csv` up to date. Safe to re-run: ledger events dedupe by `event_id`, so running this twice in a row just confirms nothing changed.

Run this regularly — the underlying Flex Query is scoped to "Last Business Day" on IBKR's side, so a missed run creates a permanent gap rather than something you can ask for later. If a run raises `TradeHistoryGapError`, see `docs/trades/ibkr_flex_api.md` for how to backfill it. All the mechanics (protocol, error codes, the ledger vs. the raw archive) are documented there too.

In [1]:
from trades.brokers.ibkr import main
from trades.config import AppConfig, IbkrFlexCredentials

credentials = IbkrFlexCredentials()  # reads IBKR_FLEX_WEB_SERVICE_TOKEN / IBKR_QUERY_ID from .env
config = AppConfig()

## Run the sync

One network round trip (SendRequest, then poll GetStatement until ready), then the ledger is updated from that single fetched statement's `<Trade>` rows.

In [2]:
result = main.sync_ibkr_account(credentials, config)
result

/Users/gabrielduguey/Documents/Perso/IBKR/src/trades/brokers/ibkr/main.py:70: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("meta").map_elements(json.loads)
with this one instead:
  + pl.col("meta").str.json_decode()

  return ledger.with_columns(pl.col("meta").map_elements(json.loads, return_dtype=pl.Object))


IbkrSyncResult(pulled_at=datetime.datetime(2026, 7, 3, 8, 2, 57), statement_from_date=datetime.date(2025, 7, 3), statement_to_date=datetime.date(2026, 7, 2), new_event_count=0, total_event_count=75)

## What's in the cache now

In [6]:
ledger = main.load_ledger(config)
first_date = ledger["event_datetime"].min().date()
last_date = ledger["event_datetime"].max().date()
print(f"{len(ledger)} ledger events cached, spanning {first_date} to {last_date}")
ledger

75 ledger events cached, spanning 2026-01-26 to 2026-07-01


event_id,event_datetime,symbol,event_type,shares,price,amount,currency,meta
str,datetime[μs],str,str,f64,f64,f64,str,object
"""ibkr:37540824085""",2026-01-26 04:06:15,"""CASH""","""DEPOSIT""",null,null,100.0,"""USD""","{'transaction_id': '37540824085', 'type': 'Deposits/Withdrawals', 'description': 'CASH RECEIPTS / ELECTRONIC FUND TRANSFERS', 'action_id': ''}"
"""ibkr:37591258188""",2026-01-27 19:32:23,"""VOO""","""BUY""",0.15,640.39,96.0585,"""USD""","{'transaction_id': '37591258188', 'trade_id': '8893138032', 'notes': 'RP'}"
"""ibkr:37591258188:fee""",2026-01-27 19:32:23,"""VOO""","""FEE""",null,null,0.960585,"""USD""","{'transaction_id': '37591258188', 'trade_id': '8893138032'}"
"""ibkr:38962164586""",2026-04-01 00:20:00,"""VOO""","""DIVIDEND""",null,null,0.28,"""USD""","{'transaction_id': '38962164586', 'type': 'Dividends', 'description': 'VOO(US9229083632) CASH DIVIDEND USD 1.8724 PER SHARE (Ordinary Dividend)', 'action_id': '164711865'}"
"""ibkr:38962164609""",2026-04-01 00:20:00,"""VOO""","""WITHHOLDING""",null,null,0.04,"""USD""","{'transaction_id': '38962164609', 'type': 'Withholding Tax', 'description': 'VOO(US9229083632) CASH DIVIDEND USD 1.8724 PER SHARE - US TAX', 'action_id': '164711865'}"
…,…,…,…,…,…,…,…,…
"""ibkr:41072945401""",2026-07-01 00:20:00,"""VOO""","""DIVIDEND""",null,null,26.77,"""USD""","{'transaction_id': '41072945401', 'type': 'Dividends', 'description': 'VOO(US9229083632) CASH DIVIDEND USD 1.9622 PER SHARE (Ordinary Dividend)', 'action_id': '168706551'}"
"""ibkr:41072945411""",2026-07-01 00:20:00,"""VOO""","""WITHHOLDING""",null,null,8.03,"""USD""","{'transaction_id': '41072945411', 'type': 'Withholding Tax', 'description': 'VOO(US9229083632) CASH DIVIDEND USD 1.9622 PER SHARE - US TAX', 'action_id': '168706551'}"
"""ibkr:41085895127""",2026-07-01 12:00:00,"""CASH""","""DEPOSIT""",null,null,2500.0,"""USD""","{'transaction_id': '41085895127', 'type': 'Deposits/Withdrawals', 'description': 'CASH RECEIPTS / ELECTRONIC FUND TRANSFERS', 'action_id': ''}"
